# This is the previous version of finalize the backend with the fabric and color swatch 

In [1]:
import torch
import gc
from google.colab import files
from diffusers import (
    StableDiffusionXLAdapterPipeline,
    T2IAdapter,
    EulerDiscreteScheduler,
    AutoencoderKL,
    UNet2DConditionModel
)
from transformers import CLIPVisionModelWithProjection
from diffusers.utils import load_image
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

# --- MEMORY CLEANUP ---
gc.collect()
torch.cuda.empty_cache()

# --- STEP 1: Uploads ---
print("1. Upload your Garment Sketch (Lineart).")
uploaded_sketch = files.upload()
sketch_path = list(uploaded_sketch.keys())[0]

print("\n2. Upload your Fabric Swatch (Pattern).")
uploaded_swatch = files.upload()
swatch_path = list(uploaded_swatch.keys())[0]

# --- STEP 2: Manual "Lightweight" Loading ---
print("🚀 Loading Models (FP16 Forced)...")

# A. Load Image Encoder manually in FP16 (Saves ~2GB RAM)
# This prevents the pipeline from trying to load the massive default version later
image_encoder = CLIPVisionModelWithProjection.from_pretrained(
    "h94/IP-Adapter",
    subfolder="sdxl_models/image_encoder",
    torch_dtype=torch.float16
)

# B. Load T2I-Adapter (Sketch Control)
adapter = T2IAdapter.from_pretrained(
    "TencentARC/t2i-adapter-sketch-sdxl-1.0",
    torch_dtype=torch.float16,
    variant="fp16"
)

# C. Load VAE (Fixed FP16)
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix",
    torch_dtype=torch.float16
)

# D. Load UNet (Lightning)
base_id = "stabilityai/stable-diffusion-xl-base-1.0"
unet = UNet2DConditionModel.from_config(base_id, subfolder="unet").to("cpu", torch.float16)
unet.load_state_dict(load_file(hf_hub_download("ByteDance/SDXL-Lightning", "sdxl_lightning_4step_unet.safetensors"), device="cpu"))

# --- STEP 3: Assemble Pipeline ---
pipe = StableDiffusionXLAdapterPipeline.from_pretrained(
    base_id,
    unet=unet,
    vae=vae,
    adapter=adapter,
    image_encoder=image_encoder, # Pass our lightweight encoder here
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
)

# --- STEP 4: Critical Memory Guards ---
pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config, timestep_spacing="trailing")

# Load the IP-Adapter weights (Fabric Texture)
# Since we already passed 'image_encoder', this won't trigger the memory spike
pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl.bin")

# aggressive offloading
pipe.enable_sequential_cpu_offload()
pipe.enable_vae_tiling()
pipe.enable_vae_slicing()

# --- STEP 5: Run ---
print("🎨 Rendering...")

# Prepare Images
sketch = load_image(sketch_path).resize((1024, 1024))
swatch = load_image(swatch_path).resize((1024, 1024))

output = pipe(
    prompt="A high-fashion tiered gown, mushroom and fern forest print, dark green silk, realistic texture, 8k",
    negative_prompt="monochrome, low res, bad anatomy, worst quality, watermark",

    # Structure (Sketch)
    image=sketch,
    adapter_conditioning_scale=0.8,

    # Texture (Fabric)
    ip_adapter_image=swatch,

    num_inference_steps=4,
    guidance_scale=0
).images[0]

output.save("final_design_fixed.png")
output.show()
print("✅ Success!")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


1. Upload your Garment Sketch (Lineart).


Saving Gemini_Generated_Image_f5zrmlf5zrmlf5zr.png to Gemini_Generated_Image_f5zrmlf5zrmlf5zr (1).png

2. Upload your Fabric Swatch (Pattern).


`torch_dtype` is deprecated! Use `dtype` instead!


Saving Gemini_Generated_Image_nee9mznee9mznee9.png to Gemini_Generated_Image_nee9mznee9mznee9 (1).png
🚀 Loading Models (FP16 Forced)...


diffusion_pytorch_model.fp16.safetensors:   0%|          | 0.00/158M [00:00<?, ?B/s]

... (2 outputs left out)

/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2213: FutureWarning: `enable_vae_tiling` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_tiling()` on a `StableDiffusionXLAdapterPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_tiling()`.
  deprecate(
/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionXLAdapterPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


🎨 Rendering...


  0%|          | 0/4 [00:00<?, ?it/s]

✅ Success!


In [26]:
import cv2
import numpy as np
from PIL import Image, ImageOps

# --- STEP 1: Canny Edge Conversion (The "Standard" Fix) ---
# Reload original sketch
original_sketch = load_image(sketch_path).resize((1024, 1024))

# Convert to numpy array
img_array = np.array(original_sketch)

# 1. Detect Edges using Canny
# This ignores messy shading and finds only the true structural lines
# (100, 200) are the sensitivity thresholds
canny_edges = cv2.Canny(img_array, 100, 200)

# 2. Dilate slightly to make sure the lines are visible to the AI
# (This prevents the "Invisible Dress" problem)
kernel = np.ones((2,2), np.uint8)
dilated_edges = cv2.dilate(canny_edges, kernel, iterations=1)

# 3. Convert to RGB (White lines on Black background)
# This format is often better for structure adapters
final_sketch_input = Image.fromarray(dilated_edges).convert("RGB")

# OPTIONAL: Preview the Canny Map to see what the AI sees
# final_sketch_input.show()

# --- STEP 2: Render with "Dress-First" Priorities ---
print("🎨 Rendering with Canny Edge Map...")

# Texture Strength: 0.6
# Keep it moderate so it doesn't overwrite the dress
pipe.set_ip_adapter_scale(0.6)

output = pipe(
    # Prompt: We weight the "Dress" higher (1.5) than the pattern
    prompt="(tiered gown, maxi dress, fashion model:1.5), (mushroom print:0.8), white background, studio lighting, highly detailed, 8k",

    # Negative Prompt: Stop the cage/wireframe look
    negative_prompt="birdcage, wireframe, see-through, vertical bars, stripes, plaid, mosaic, forest, texture only, seamless pattern",

    # Use the CANNY map
    image=final_sketch_input,

    # Structure Strength: 0.7
    # Canny maps are clean, so we can trust them with a higher weight
    # without getting the "Cage" effect.
    adapter_conditioning_scale=0.7,

    ip_adapter_image=swatch,

    num_inference_steps=4,
    guidance_scale=0
).images[0]

output.save("final_design_canny.png")
output.show()
print("✅ Done! Used Canny Edges for perfect structure.")

🎨 Rendering with Canny Edge Map...


  0%|          | 0/4 [00:00<?, ?it/s]

✅ Done! Used Canny Edges for perfect structure.
